# 🚗 RoadLoad Model — Development Notebook

## Objetivo
Descrever e implementar o subsistema `RoadLoadModel` como um bloco macro de engenharia:

- **Entrada:** baseline + cenário + contexto de resolução
- **Saída:** modelo de resistência longitudinal intercambiável
- **Uso futuro:** VDE, benchmark, simulação, operating points, diagnóstico

---

## Arquitetura Macro do Bloco

### Entradas
- baseline agregado (A/B/C, massa, legislação, categoria)
- informações opcionais de componentes
- modificadores de cenário
- regras de resolução

### Pipeline
1. Intake / Normalization
2. Baseline Resolution
3. Build Component Set
4. Apply Component Changes
5. Synthesize Equivalent ABC
6. Build RoadLoadModel
7. Compute VDE

### Saídas
- `roadload_model.force(v)`
- `roadload_model.power(v)`
- coeficientes equivalentes A/B/C
- breakdown por componente
- proveniência
- warnings

## 📥 1. Inputs do Sistema

Este bloco define o contrato de entrada do subsistema `RoadLoadModel`.

### Objetivo
Padronizar tudo que pode entrar no modelo, independentemente de vir de:
- baseline do DB
- input manual
- ETL
- cenário derivado
- futuro estimador

### Grupos de entrada

#### 1.1 Baseline Input
Dados físicos ou regulatórios do veículo base:
- `baseline_id`
- `A`, `B`, `C`
- `mass_kg`
- `legislation`
- `category`
- `source`

#### 1.2 Operating Modifiers
Mudanças globais no cenário:
- `delta_mass_kg`
- `trailer`
- `target_legislation`

#### 1.3 Component Changes
Mudanças por subsistema:
- `tire_change`
- `aero_change`
- `transmission_change`
- `axle_change`
- `brake_change`
- `hubs_change`

#### 1.4 Resolution Options
Regras que controlam como o sistema resolve parâmetros faltantes:
- `allow_estimation`
- `use_defaults`
- `inherit_from_baseline`

### Saída esperada deste bloco
Um `RoadLoadRequest` validado e normalizado, pronto para entrar no pipeline:
- intake
- baseline resolution
- component set build
- component changes
- synthesis
- roadload model

In [1]:
from dataclasses import dataclass, field, asdict
from typing import Optional, Dict, Any


@dataclass
class BaselineInput:
    baseline_id: Optional[int] = None
    A: Optional[float] = None
    B: Optional[float] = None
    C: Optional[float] = None
    mass_kg: Optional[float] = None
    legislation: Optional[str] = None
    category: Optional[str] = None
    source: str = "unknown"


@dataclass
class OperatingModifiers:
    delta_mass_kg: float = 0.0
    trailer: bool = False
    target_legislation: Optional[str] = None


@dataclass
class ComponentChange:
    mode: str = "inherit"   # inherit | replace | delta_abc | delta_cda | improve
    A: float = 0.0
    B: float = 0.0
    C: float = 0.0
    delta_cda_m2: float = 0.0
    improve_pct: float = 0.0
    meta: Dict[str, Any] = field(default_factory=dict)


@dataclass
class ComponentChanges:
    tire: Optional[ComponentChange] = None
    aero: Optional[ComponentChange] = None
    transmission: Optional[ComponentChange] = None
    axle: Optional[ComponentChange] = None
    brakes: Optional[ComponentChange] = None
    hubs: Optional[ComponentChange] = None
    parasitic: Optional[ComponentChange] = None


@dataclass
class ResolutionOptions:
    allow_estimation: bool = False
    use_defaults: bool = True
    inherit_from_baseline: bool = True


@dataclass
class RoadLoadRequest:
    baseline: BaselineInput
    operating: OperatingModifiers = field(default_factory=OperatingModifiers)
    components: ComponentChanges = field(default_factory=ComponentChanges)
    options: ResolutionOptions = field(default_factory=ResolutionOptions)
    extra: Dict[str, Any] = field(default_factory=dict)

In [2]:
def normalize_roadload_request(request: RoadLoadRequest) -> RoadLoadRequest:
    """
    Normaliza tipos e defaults mínimos.
    Não resolve física ainda. Só limpa o contrato.
    """
    b = request.baseline
    op = request.operating
    c = request.components
    o = request.options

    # baseline
    if b.A is not None:
        b.A = float(b.A)
    if b.B is not None:
        b.B = float(b.B)
    if b.C is not None:
        b.C = float(b.C)
    if b.mass_kg is not None:
        b.mass_kg = float(b.mass_kg)

    if b.legislation is not None:
        b.legislation = str(b.legislation).strip().upper()

    if b.category is not None:
        b.category = str(b.category).strip().upper()

    # operating modifiers
    op.delta_mass_kg = float(op.delta_mass_kg)
    op.trailer = bool(op.trailer)

    if op.target_legislation is not None:
        op.target_legislation = str(op.target_legislation).strip().upper()

    # component changes
    for comp_name in ["tire", "aero", "transmission", "axle", "brakes", "hubs", "parasitic"]:
        comp = getattr(c, comp_name)
        if comp is not None:
            comp.A = float(comp.A)
            comp.B = float(comp.B)
            comp.C = float(comp.C)
            comp.delta_cda_m2 = float(comp.delta_cda_m2)
            comp.improve_pct = float(comp.improve_pct)
            comp.mode = str(comp.mode).strip().lower()

    # options
    o.allow_estimation = bool(o.allow_estimation)
    o.use_defaults = bool(o.use_defaults)
    o.inherit_from_baseline = bool(o.inherit_from_baseline)

    return request

In [3]:
req = RoadLoadRequest(
    baseline=BaselineInput(
        baseline_id=101,
        A=120.0,
        B=0.02,
        C=0.011,
        mass_kg=1550.0,
        legislation="EPA",
        category="MIDSIZE",
        source="measured"
    ),
    operating=OperatingModifiers(
        delta_mass_kg=80.0,
        trailer=False,
        target_legislation=None
    ),
    components=ComponentChanges(
        tire=ComponentChange(mode="improve", improve_pct=5.0),
        aero=ComponentChange(mode="delta_cda", delta_cda_m2=0.03),
        brakes=ComponentChange(mode="delta_abc", A=1.0, B=-0.002, C=0.0)
    ),
    options=ResolutionOptions(
        allow_estimation=False,
        use_defaults=True,
        inherit_from_baseline=True
    )
)

req = normalize_roadload_request(req)
asdict(req)

{'baseline': {'baseline_id': 101,
  'A': 120.0,
  'B': 0.02,
  'C': 0.011,
  'mass_kg': 1550.0,
  'legislation': 'EPA',
  'category': 'MIDSIZE',
  'source': 'measured'},
 'operating': {'delta_mass_kg': 80.0,
  'trailer': False,
  'target_legislation': None},
 'components': {'tire': {'mode': 'improve',
   'A': 0.0,
   'B': 0.0,
   'C': 0.0,
   'delta_cda_m2': 0.0,
   'improve_pct': 5.0,
   'meta': {}},
  'aero': {'mode': 'delta_cda',
   'A': 0.0,
   'B': 0.0,
   'C': 0.0,
   'delta_cda_m2': 0.03,
   'improve_pct': 0.0,
   'meta': {}},
  'transmission': None,
  'axle': None,
  'brakes': {'mode': 'delta_abc',
   'A': 1.0,
   'B': -0.002,
   'C': 0.0,
   'delta_cda_m2': 0.0,
   'improve_pct': 0.0,
   'meta': {}},
  'hubs': None,
  'parasitic': None},
 'options': {'allow_estimation': False,
  'use_defaults': True,
  'inherit_from_baseline': True},
 'extra': {}}

## 🔄 2.1 Intake / Normalization

### Objetivo
Transformar o `RoadLoadRequest` em uma entrada consistente, com tipos corretos e campos mínimos padronizados.

### Funções deste passo
- Validar tipos
- Garantir defaults mínimos
- Padronizar strings e campos opcionais
- Evitar que o restante do pipeline lide com payload “sujo”

### Saída
Um `RoadLoadRequest` limpo e estável.

---

## 🔄 2.2 Baseline Resolution

### Objetivo
Definir o ponto de partida físico do roadload.

### Regras iniciais
1. Se `A/B/C/mass_kg` vierem medidos no request, usar esses valores
2. Se não vierem, mas existir baseline no DB, usar baseline
3. Se vier mistura, priorizar valor medido
4. Se faltar algo essencial nesta primeira versão, gerar erro

### Saída
Um `ResolvedBaseline` pronto para receber deltas

### Observação
Nesta versão ainda não:
- estima parâmetros automaticamente
- remapeia legislação
- corrige transmissão neutra
- usa decomposition completa de componentes medidos

O próximo passo será:
- construir o `ComponentSet`
- aplicar mudanças por componente
- sintetizar o ABC equivalente final

In [4]:
from dataclasses import dataclass, asdict
from typing import Optional, Dict, Any


@dataclass
class ResolvedBaseline:
    A: float
    B: float
    C: float
    mass_kg: float
    legislation: Optional[str] = None
    category: Optional[str] = None
    source_map: Dict[str, str] = None
    warnings: list[str] = None

In [5]:
def resolve_baseline(
    request: RoadLoadRequest,
    baseline_record: Optional[Dict[str, Any]] = None
) -> ResolvedBaseline:
    """
    Resolve o baseline físico inicial.

    Prioridade:
    1. Valor medido no request
    2. Valor vindo do baseline_record
    3. Erro, se essencial faltar
    """
    b = request.baseline
    warnings = []
    source_map = {}

    def pick_value(field_name: str):
        request_value = getattr(b, field_name, None)
        baseline_value = None if baseline_record is None else baseline_record.get(field_name)

        if request_value is not None:
            source_map[field_name] = "request_measured"
            return request_value

        if baseline_value is not None and request.options.inherit_from_baseline:
            source_map[field_name] = "baseline_inherited"
            return baseline_value

        source_map[field_name] = "missing"
        return None

    A = pick_value("A")
    B = pick_value("B")
    C = pick_value("C")
    mass_kg = pick_value("mass_kg")

    legislation = pick_value("legislation")
    category = pick_value("category")

    missing = [k for k, v in {"A": A, "B": B, "C": C, "mass_kg": mass_kg}.items() if v is None]
    if missing:
        raise ValueError(f"Baseline incompleto. Campos obrigatórios faltando: {missing}")

    return ResolvedBaseline(
        A=float(A),
        B=float(B),
        C=float(C),
        mass_kg=float(mass_kg),
        legislation=legislation,
        category=category,
        source_map=source_map,
        warnings=warnings
    )

In [6]:
req = RoadLoadRequest(
    baseline=BaselineInput(
        baseline_id=101,
        A=120,
        B=0.02,
        C=0.011,
        mass_kg=1550,
        legislation="EPA",
        category="MIDSIZE",
        source="measured"
    ),
    operating=OperatingModifiers(
        delta_mass_kg=80,
        trailer=False,
        target_legislation=None
    ),
    components=ComponentChanges(
        tire=ComponentChange(mode="improve", improve_pct=5.0),
        aero=ComponentChange(mode="delta_cda", delta_cda_m2=0.03)
    ),
    options=ResolutionOptions(
        allow_estimation=False,
        use_defaults=True,
        inherit_from_baseline=True
    )
)

req = normalize_roadload_request(req)
baseline = resolve_baseline(req)
asdict(baseline)

{'A': 120.0,
 'B': 0.02,
 'C': 0.011,
 'mass_kg': 1550.0,
 'legislation': 'EPA',
 'category': 'MIDSIZE',
 'source_map': {'A': 'request_measured',
  'B': 'request_measured',
  'C': 'request_measured',
  'mass_kg': 'request_measured',
  'legislation': 'request_measured',
  'category': 'request_measured'},
 'warnings': []}

In [7]:
req2 = RoadLoadRequest(
    baseline=BaselineInput(
        baseline_id=102,
        A=None,
        B=None,
        C=None,
        mass_kg=1650,
        legislation="EPA",
        category=None,
        source="mixed"
    ),
    operating=OperatingModifiers(
        delta_mass_kg=50,
        trailer=False,
        target_legislation=None
    ),
    components=ComponentChanges(),
    options=ResolutionOptions(inherit_from_baseline=True)
)

baseline_record = {
    "A": 135.0,
    "B": 0.03,
    "C": 0.012,
    "mass_kg": 1600.0,
    "legislation": "EPA",
    "category": "SUV"
}

req2 = normalize_roadload_request(req2)
baseline2 = resolve_baseline(req2, baseline_record=baseline_record)
asdict(baseline2)

{'A': 135.0,
 'B': 0.03,
 'C': 0.012,
 'mass_kg': 1650.0,
 'legislation': 'EPA',
 'category': 'SUV',
 'source_map': {'A': 'baseline_inherited',
  'B': 'baseline_inherited',
  'C': 'baseline_inherited',
  'mass_kg': 'request_measured',
  'legislation': 'request_measured',
  'category': 'baseline_inherited'},
 'warnings': []}

In [9]:
from dataclasses import dataclass, field
from typing import Dict, Any


@dataclass
class RoadLoadComponent:
    name: str
    A: float = 0.0
    B: float = 0.0
    C: float = 0.0
    source: str = "unknown"
    meta: Dict[str, Any] = field(default_factory=dict)

    def as_dict(self):
        return {
            "name": self.name,
            "A": self.A,
            "B": self.B,
            "C": self.C,
            "source": self.source,
            "meta": self.meta,
        }

In [12]:
from dataclasses import dataclass, field
from typing import Dict, Optional


@dataclass
class ComponentSet:
    components: Dict[str, RoadLoadComponent] = field(default_factory=dict)

    def add(self, component: RoadLoadComponent):
        self.components[component.name] = component

    def get(self, name: str) -> Optional[RoadLoadComponent]:
        return self.components.get(name)

    def names(self):
        return list(self.components.keys())

    def as_table(self):
        return [comp.as_dict() for comp in self.components.values()]

In [13]:
component_set = ComponentSet()

component_set.add(
    RoadLoadComponent(
        name="roadload_total",
        A=120.0,
        B=0.02,
        C=0.011,
        source="baseline_resolved"
    )
)

component_set

ComponentSet(components={'roadload_total': RoadLoadComponent(name='roadload_total', A=120.0, B=0.02, C=0.011, source='baseline_resolved', meta={})})

In [14]:
def build_component_set_from_baseline(baseline: ResolvedBaseline) -> ComponentSet:
    component_set = ComponentSet()

    component_set.add(
        RoadLoadComponent(
            name="roadload_total",
            A=baseline.A,
            B=baseline.B,
            C=baseline.C,
            source="baseline_resolved",
            meta={
                "legislation": baseline.legislation,
                "category": baseline.category,
                "source_map": baseline.source_map,
            }
        )
    )

    return component_set

In [15]:
req = RoadLoadRequest(
    baseline=BaselineInput(
        baseline_id=101,
        A=120.0,
        B=0.02,
        C=0.011,
        mass_kg=1550.0,
        legislation="EPA",
        category="MIDSIZE",
        source="measured"
    ),
    operating=OperatingModifiers(
        delta_mass_kg=80.0,
        trailer=False,
        target_legislation=None
    ),
    components=ComponentChanges(),
    options=ResolutionOptions(
        allow_estimation=False,
        use_defaults=True,
        inherit_from_baseline=True
    )
)

req = normalize_roadload_request(req)
baseline = resolve_baseline(req)
component_set = build_component_set_from_baseline(baseline)

component_set.as_table()

[{'name': 'roadload_total',
  'A': 120.0,
  'B': 0.02,
  'C': 0.011,
  'source': 'baseline_resolved',
  'meta': {'legislation': 'EPA',
   'category': 'MIDSIZE',
   'source_map': {'A': 'request_measured',
    'B': 'request_measured',
    'C': 'request_measured',
    'mass_kg': 'request_measured',
    'legislation': 'request_measured',
    'category': 'request_measured'}}}]

In [16]:
def apply_component_change(base_comp: RoadLoadComponent, change: ComponentChange) -> RoadLoadComponent:
    if change is None or change.mode == "inherit":
        return base_comp

    A = base_comp.A
    B = base_comp.B
    C = base_comp.C

    # 1) replace total
    if change.mode == "replace":
        return RoadLoadComponent(
            name=base_comp.name,
            A=change.A,
            B=change.B,
            C=change.C,
            source="component_replaced",
            meta=change.meta
        )

    # 2) delta direto em ABC
    if change.mode == "delta_abc":
        return RoadLoadComponent(
            name=base_comp.name,
            A=A + change.A,
            B=B + change.B,
            C=C + change.C,
            source="component_delta_abc",
            meta=change.meta
        )

    # 3) delta CdA -> atua em C
    if change.mode == "delta_cda":
        delta_C = cdA_to_C(change.delta_cda_m2)
        return RoadLoadComponent(
            name=base_comp.name,
            A=A,
            B=B,
            C=C + delta_C,
            source="component_delta_cda",
            meta=change.meta
        )

    # 4) improvement percentual
    if change.mode == "improve":
        factor = 1.0 - (change.improve_pct / 100.0)
        return RoadLoadComponent(
            name=base_comp.name,
            A=A * factor,
            B=B * factor,
            C=C * factor,
            source="component_improved",
            meta=change.meta
        )

    raise ValueError(f"Modo desconhecido: {change.mode}")

---

## Testing Step

In [17]:
base_comp = RoadLoadComponent(
    name="roadload_total",
    A=120.0,
    B=0.02,
    C=0.011,
    source="baseline_resolved"
)

change = ComponentChange(mode="inherit")

apply_component_change(base_comp, change)

RoadLoadComponent(name='roadload_total', A=120.0, B=0.02, C=0.011, source='baseline_resolved', meta={})

In [18]:
change = ComponentChange(
    mode="delta_abc",
    A=5.0,
    B=0.001,
    C=0.0002
)

apply_component_change(base_comp, change)

RoadLoadComponent(name='roadload_total', A=125.0, B=0.021, C=0.0112, source='component_delta_abc', meta={})

In [19]:
change = ComponentChange(
    mode="improve",
    improve_pct=5.0
)

apply_component_change(base_comp, change)

RoadLoadComponent(name='roadload_total', A=114.0, B=0.019, C=0.01045, source='component_improved', meta={})

In [23]:
def apply_component_changes(component_set: ComponentSet, request: RoadLoadRequest) -> ComponentSet:
    updated = ComponentSet(components=dict(component_set.components))

    target_name = "roadload_total"
    base_comp = updated.get(target_name)

    if base_comp is None:
        raise ValueError("ComponentSet sem 'roadload_total'.")

    for comp_name in ["tire", "aero", "transmission", "axle", "brakes", "hubs", "parasitic"]:
        change = getattr(request.components, comp_name)

        if change is not None:
            new_comp = apply_component_change(base_comp, change)
            updated.add(new_comp)
            base_comp = updated.get(target_name)

    return updated

In [26]:
def cdA_to_C(delta_cda_m2: float, rho: float = 1.2) -> float:
    """
    Converte delta CdA [m²] para delta C [N/kph²]
    """
    return 0.5 * rho * delta_cda_m2 * (1 / 3.6) ** 2

In [27]:
req = RoadLoadRequest(
    baseline=BaselineInput(
        baseline_id=101,
        A=120.0,
        B=0.02,
        C=0.011,
        mass_kg=1550.0,
        legislation="EPA",
        category="MIDSIZE",
        source="measured"
    ),
    operating=OperatingModifiers(
        delta_mass_kg=80.0,
        trailer=False,
        target_legislation=None
    ),
    components=ComponentChanges(
        tire=ComponentChange(mode="improve", improve_pct=5.0),
        aero=ComponentChange(mode="delta_cda", delta_cda_m2=0.03),
        brakes=ComponentChange(mode="delta_abc", A=1.0, B=-0.002, C=0.0)
    ),
    options=ResolutionOptions(
        allow_estimation=False,
        use_defaults=True,
        inherit_from_baseline=True
    )
)

req = normalize_roadload_request(req)
baseline = resolve_baseline(req)
component_set = build_component_set_from_baseline(baseline)

updated_component_set = apply_component_changes(component_set, req)

updated_component_set.as_table()

[{'name': 'roadload_total',
  'A': 115.0,
  'B': 0.017,
  'C': 0.011838888888888888,
  'source': 'component_delta_abc',
  'meta': {}}]

In [28]:
from dataclasses import dataclass, field
from typing import List, Dict, Any


@dataclass
class EquivalentABC:
    A: float
    B: float
    C: float
    mass_kg: float
    component_table: List[Dict[str, Any]]
    warnings: List[str] = field(default_factory=list)

In [29]:
def synthesize_equivalent_abc(
    component_set: ComponentSet,
    baseline: ResolvedBaseline,
    request: RoadLoadRequest
) -> EquivalentABC:

    # soma todos os componentes do conjunto
    A = sum(comp.A for comp in component_set.components.values())
    B = sum(comp.B for comp in component_set.components.values())
    C = sum(comp.C for comp in component_set.components.values())

    # massa ainda vem do baseline + modificador operacional
    mass_kg = baseline.mass_kg + request.operating.delta_mass_kg

    warnings = []

    if request.operating.trailer:
        warnings.append("Trailer flag active, but trailer physics is not implemented yet.")

    if (
        request.operating.target_legislation is not None
        and request.operating.target_legislation != baseline.legislation
    ):
        warnings.append("Target legislation requested, but test-mass remapping is not implemented yet.")

    return EquivalentABC(
        A=A,
        B=B,
        C=C,
        mass_kg=mass_kg,
        component_table=component_set.as_table(),
        warnings=warnings
    )

In [30]:
req = RoadLoadRequest(
    baseline=BaselineInput(
        baseline_id=101,
        A=120.0,
        B=0.02,
        C=0.011,
        mass_kg=1550.0,
        legislation="EPA",
        category="MIDSIZE",
        source="measured"
    ),
    operating=OperatingModifiers(
        delta_mass_kg=80.0,
        trailer=False,
        target_legislation=None
    ),
    components=ComponentChanges(
        tire=ComponentChange(mode="improve", improve_pct=5.0),
        aero=ComponentChange(mode="delta_cda", delta_cda_m2=0.03),
        brakes=ComponentChange(mode="delta_abc", A=1.0, B=-0.002, C=0.0)
    ),
    options=ResolutionOptions(
        allow_estimation=False,
        use_defaults=True,
        inherit_from_baseline=True
    )
)

req = normalize_roadload_request(req)
baseline = resolve_baseline(req)
component_set = build_component_set_from_baseline(baseline)
updated_component_set = apply_component_changes(component_set, req)
equiv = synthesize_equivalent_abc(updated_component_set, baseline, req)

equiv

EquivalentABC(A=115.0, B=0.017, C=0.011838888888888888, mass_kg=1630.0, component_table=[{'name': 'roadload_total', 'A': 115.0, 'B': 0.017, 'C': 0.011838888888888888, 'source': 'component_delta_abc', 'meta': {}}], warnings=[])

In [31]:
class RoadLoadModel:
    def __init__(self, A: float, B: float, C: float, mass_kg: float):
        self.A = A
        self.B = B
        self.C = C
        self.mass_kg = mass_kg

    def force(self, v_kph: float) -> float:
        """
        Força resistiva total [N]
        v em km/h
        """
        return self.A + self.B * v_kph + self.C * (v_kph ** 2)

    def power(self, v_kph: float) -> float:
        """
        Potência resistiva [W]
        """
        v_mps = v_kph / 3.6
        F = self.force(v_kph)
        return F * v_mps

    def summary(self):
        return {
            "A": self.A,
            "B": self.B,
            "C": self.C,
            "mass_kg": self.mass_kg
        }

In [32]:
model = RoadLoadModel(
    A=equiv.A,
    B=equiv.B,
    C=equiv.C,
    mass_kg=equiv.mass_kg
)

model.summary()

{'A': 115.0, 'B': 0.017, 'C': 0.011838888888888888, 'mass_kg': 1630.0}